In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-02 18:09:38--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.1s    

2026-07-02 18:09:39 (10.5 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
print(f"Length of dataset in characters: {len(text):,}")

Length of dataset in characters: 1,115,394


In [13]:
!pip install datasets
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("PrimeIntellect/SYNTHETIC-2-SFT-verified")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 12.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 18.0 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: fsspec━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/13 [pyarrow]
    Found existing installation: fsspec 2026.6.0━━━━━━━━━━━━━━  1/13 [pyarrow]
    Uninstalling fsspec-2026.6.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/13 [pyarrow]
      Successfully uninstalled fsspec-2026.6.0━━━━━━━━━━━━━━━━  1/13 [pyarrow]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [datasets]/13 [datasets]ess]


README.md:   0%|          | 0.00/2.00k [00:00<?, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/314M [00:00<?, ?B/s]

data/train-00001-of-00006.parquet:   0%|          | 0.00/196M [00:00<?, ?B/s]

data/train-00002-of-00006.parquet:   0%|          | 0.00/196M [00:00<?, ?B/s]

data/train-00003-of-00006.parquet:   0%|          | 0.00/175M [00:00<?, ?B/s]

data/train-00004-of-00006.parquet:   0%|          | 0.00/67.6M [00:00<?, ?B/s]

data/train-00005-of-00006.parquet:   0%|          | 0.00/191M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/104913 [00:00<?, ? examples/s]

In [15]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['problem_id', 'task_type', 'reward', 'messages'],
        num_rows: 104913
    })
})


In [18]:
train_ds = ds["train"]
def format_Chat_turn(row):
    text_block = ""
    for message in row['messages']:
        role = message['role']
        content = message['content']
        if role == 'user':
            text_block += f"<|user|>: {content}\n"
        elif role == 'assistant':
            text_block += f"<|assistant|>: {content}\n"
    
    text_block += "<|endoftext|>\n\n"
    return text_block

split_index = int(len(train_ds) * 0.8)

In [19]:


from tqdm import tqdm


def save_to_file(dataset, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for row in tqdm(dataset):
            formatted_text = format_Chat_turn(row)
            f.write(formatted_text)

save_to_file(train_ds.select(range(split_index)), 'train.txt')
save_to_file(train_ds.select(range(split_index, len(train_ds))), 'val.txt')

print("Data saved to train.txt and val.txt")

100%|██████████| 20983/20983 [00:01<00:00, 14616.45it/s]

Data saved to train.txt and val.txt


In [ ]:
with open('data/train.txt', 'r', encoding='utf-8') as f:
    train_text = f.read()
with open('data/val.txt', 'r', encoding='utf-8') as f:
    val_text = f.read()

In [23]:
chars = sorted(list(set(train_text + val_text)))
vocab_size = len(chars)
print(f"Vocab size: {vocab_size}")

Vocab size: 5530


In [24]:
stoi = { ch:i for i,ch in enumerate(chars)}
itos = { i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder:

encode("s=ome text  to encode")

[97,
 43,
 93,
 91,
 83,
 14,
 98,
 83,
 102,
 98,
 14,
 14,
 98,
 93,
 14,
 83,
 92,
 81,
 93,
 82,
 83]

In [25]:
encoded_train = encode(train_text)
encoded_val = encode(val_text)

In [26]:

import torch
import torch.nn as nn

train_ids = torch.tensor(encoded_train, dtype=torch.long)
val_ids = torch.tensor(encoded_val, dtype=torch.long)

In [29]:
def tofile(tensor, filename):
    tensor.numpy().tofile(filename)

# Use the function you defined
tofile(train_ids, 'train_ids.bin')
tofile(val_ids, 'val_ids.bin')

In [30]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [31]:
import pickle
metadata = {
    'vocab_size': vocab_size,
    'itos': itos,
    'stoi': stoi,
}
with open('metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

In [ ]:
from torch.nn import ffunctional as F
import numpy as np

batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = device
eval_iters = 200
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.1


train_data = np.memmap('train_ids.bin', dtype=np.uint16, mode='r')
val_data = np.memmap('val_ids.bin', dtype=np.uint16, mode='r')


ImportError: cannot import name 'ffunctional' from 'torch.functional' (/opt/homebrew/Caskroom/miniconda/base/envs/face/lib/python3.11/site-packages/torch/functional.py)